# Learning Compact Representations via Intrinsic Dimension Regularization (IDRR)

Reimplementation of the GRaM @ ICLR 2026 paper. This notebook reproduces every experiment in the paper end to end:

1. Soft effective rank (Definition 1) and the two-sided IDRR loss (Eq. 2) with the data-driven target rank (Eq. 3).
2. Seven methods: Standard, Weight Decay, Dropout, Jacobian Reg., IDRR, IDRR-Adaptive, IDRR+Dropout.
3. Four datasets: Synthetic 8-d manifold in R^100, MNIST, Fashion-MNIST, CIFAR-10 (10k train / 2k test subsets).
4. MLP results (Table 1) and CNN results (Table 2), mean ± 95% CI over 5 seeds (3 for CIFAR-10).
5. Paired t-tests (Table 3), PCA skeletons (Figure 1), singular value decay (Figure 2), training dynamics (Figures 3, 5, 6, 7), rank bar chart (Figure 4).
6. Ablations on lambda and target rank (Table 4) and the bottleneck baselines (Table 5).

Set `RUN_MODE` below. `"smoke"` runs everything with 1 seed and a handful of epochs so you can check the pipeline in a few minutes. `"full"` reproduces the paper's protocol (500 max epochs with early stopping, all seeds). On a Colab T4 the full MLP sweep is roughly 1.5 to 2 hours and the CNN sweep another 3 to 4 hours, so the CNN section is gated by `RUN_CNN`.

Everything is deterministic given the seed list. Results land in `results/` as CSV and PNG so they survive a runtime restart if you mount Drive.

In [ ]:
RUN_MODE = "full"          # "smoke" or "full"
RUN_CNN = True             # Table 2 takes the longest; set False to skip
SAVE_TO_DRIVE = False      # set True in Colab to persist results/ to Google Drive

import os, math, time, json, random, itertools
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

try:
    import torchvision
    from torchvision import datasets, transforms
except ImportError:
    os.system("pip install -q torchvision")
    import torchvision
    from torchvision import datasets, transforms

import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/idrr_results"
else:
    OUT = "results"
os.makedirs(OUT, exist_ok=True)

if RUN_MODE == "smoke":
    SEEDS = {"synthetic": [0], "mnist": [0], "fashion_mnist": [0], "cifar10": [0]}
    MAX_EPOCHS, PATIENCE = 6, 3
else:
    SEEDS = {"synthetic": [0, 1, 2, 3, 4], "mnist": [0, 1, 2, 3, 4],
             "fashion_mnist": [0, 1, 2, 3, 4], "cifar10": [0, 1, 2]}
    MAX_EPOCHS, PATIENCE = 500, 15

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Soft effective rank and the IDRR loss

For a batch of representations Z (m x D), center it, take singular values, normalise them to a distribution p, and return exp(H(p)). This is Roy and Vetterli's effective rank and it is differentiable through `torch.linalg.svdvals`.

The loss is two-sided: a penalty when erank exceeds d_max = d_target + margin, and a lighter penalty (alpha = 0.3) when it falls below d_min = max(d_target / 2, 3).

In [ ]:
def soft_effective_rank(z: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """exp(Shannon entropy of normalised singular values) of the centred matrix z."""
    zc = z - z.mean(dim=0, keepdim=True)
    s = torch.linalg.svdvals(zc.float())
    p = s / (s.sum() + eps)
    p = p[p > eps]
    return torch.exp(-(p * torch.log(p)).sum())


def idrr_loss(erank: torch.Tensor, d_target: float, margin: float = 2.0, alpha: float = 0.3) -> torch.Tensor:
    d_max = d_target + margin
    d_min = max(d_target / 2.0, 3.0)
    return F.relu(erank - d_max) + alpha * F.relu(d_min - erank)


@torch.no_grad()
def data_driven_target_rank(x: torch.Tensor, n_classes: int, max_points: int = 4000) -> float:
    """Eq. 3: sqrt((k - 1) * erank(X)), erank of the (flattened) input data."""
    xf = x.reshape(x.shape[0], -1)
    if xf.shape[0] > max_points:
        idx = torch.randperm(xf.shape[0], generator=torch.Generator().manual_seed(0))[:max_points]
        xf = xf[idx]
    er = soft_effective_rank(xf.to(DEVICE)).item()
    return float(math.sqrt((n_classes - 1) * er))


# sanity checks
_z = torch.randn(256, 64)
print("erank of Gaussian noise (should be near 64):", round(soft_effective_rank(_z).item(), 2))
_low = torch.randn(256, 4) @ torch.randn(4, 64)
print("erank of rank-4 matrix (should be near 4):", round(soft_effective_rank(_low).item(), 2))

## 2. Datasets

Synthetic: 5000 training points on an 8-dimensional manifold embedded in R^100 with 5 classes. Class centres live in the 8-d latent space; the embedding is a fixed random nonlinear map followed by a random rotation so the manifold is curved, then isotropic ambient noise. The exact generator from the original run was lost with the code, so this one is tuned to land in the same regime (about 70 to 77% test accuracy for the baselines); absolute effective ranks on this dataset can differ from the paper's table while the relative ordering of methods holds.

MNIST, Fashion-MNIST, CIFAR-10: a stratified 10,000-example training subset and a 2,000-example test subset, as in the paper. A 10% validation slice of the training subset drives early stopping. MLPs see flattened inputs (784 or 3072); CNNs see images.

In [ ]:
def make_synthetic(n_train=5000, n_test=1000, d_latent=8, D=100, n_classes=5, seed=0):
    g = torch.Generator().manual_seed(1000 + seed)
    centres = torch.randn(n_classes, d_latent, generator=g) * 1.2
    W1 = torch.randn(d_latent, 64, generator=g) / math.sqrt(d_latent)
    W2 = torch.randn(64, D, generator=g) / math.sqrt(64)
    Q, _ = torch.linalg.qr(torch.randn(D, D, generator=g))

    def sample(n):
        y = torch.randint(0, n_classes, (n,), generator=g)
        z = centres[y] + 1.2 * torch.randn(n, d_latent, generator=g)
        h = torch.tanh(z @ W1)
        x = (torch.sin(h @ W2) + h @ W2 * 0.5) @ Q
        x = x + 1.0 * torch.randn(n, D, generator=g)
        return x.float(), y
    xtr, ytr = sample(n_train)
    xte, yte = sample(n_test)
    mu, sd = xtr.mean(0), xtr.std(0) + 1e-6
    return (xtr - mu) / sd, ytr, (xte - mu) / sd, yte


def stratified_subset(y: torch.Tensor, n: int, seed: int) -> torch.Tensor:
    g = torch.Generator().manual_seed(seed)
    classes = y.unique()
    per = n // len(classes)
    idx = []
    for c in classes:
        ci = (y == c).nonzero().squeeze(1)
        idx.append(ci[torch.randperm(len(ci), generator=g)[:per]])
    return torch.cat(idx)


_TV_CACHE = {}

def load_torchvision(name: str, n_train=10000, n_test=2000, seed=0):
    key = (name, seed)
    if key in _TV_CACHE:
        return _TV_CACHE[key]
    cls = {"mnist": datasets.MNIST, "fashion_mnist": datasets.FashionMNIST, "cifar10": datasets.CIFAR10}[name]
    tr = cls("./data", train=True, download=True)
    te = cls("./data", train=False, download=True)
    def to_tensor(ds):
        x = torch.as_tensor(np.array(ds.data)).float() / 255.0
        y = torch.as_tensor(np.array(ds.targets)).long()
        if x.dim() == 3:
            x = x.unsqueeze(1)              # N,1,28,28
        else:
            x = x.permute(0, 3, 1, 2)       # N,3,32,32
        return x, y
    xtr, ytr = to_tensor(tr)
    xte, yte = to_tensor(te)
    itr = stratified_subset(ytr, n_train, seed)
    ite = stratified_subset(yte, n_test, seed + 7)
    xtr, ytr, xte, yte = xtr[itr], ytr[itr], xte[ite], yte[ite]
    mean = xtr.mean(dim=(0, 2, 3), keepdim=True)
    std = xtr.std(dim=(0, 2, 3), keepdim=True) + 1e-6
    out = ((xtr - mean) / std, ytr, (xte - mean) / std, yte)
    _TV_CACHE[key] = out
    return out


@dataclass
class DataBundle:
    name: str
    xtr: torch.Tensor; ytr: torch.Tensor
    xva: torch.Tensor; yva: torch.Tensor
    xte: torch.Tensor; yte: torch.Tensor
    n_classes: int
    d_target: float
    in_shape: tuple


def get_data(name: str, seed: int, flatten: bool) -> DataBundle:
    if name == "synthetic":
        xtr, ytr, xte, yte = make_synthetic(seed=seed)
        n_classes = 5
        if not flatten:   # the CNN sees the 100-d point as a 1 x 10 x 10 image
            xtr, xte = xtr.reshape(-1, 1, 10, 10), xte.reshape(-1, 1, 10, 10)
    else:
        xtr, ytr, xte, yte = load_torchvision(name, seed=seed)
        n_classes = 10
    if flatten:
        xtr, xte = xtr.reshape(len(xtr), -1), xte.reshape(len(xte), -1)
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(xtr), generator=g)
    n_val = len(xtr) // 10
    va, tr = perm[:n_val], perm[n_val:]
    d_target = data_driven_target_rank(xtr, n_classes)
    return DataBundle(name, xtr[tr], ytr[tr], xtr[va], ytr[va], xte, yte, n_classes, d_target, tuple(xtr.shape[1:]))

## 3. Models

MLP: hidden [256, 128] (or [512, 256, 128] for CIFAR-10), BatchNorm after each hidden layer, ReLU, representation dimension 64 (128 for CIFAR-10), then a linear head. Dropout, when used, sits after each hidden activation.

CNN: three conv blocks with BN and ReLU, max-pooling, global average pooling, a fully connected representation layer (64, or 128 for CIFAR-10) with ReLU, then a linear head. Exactly the backbone listed in Appendix C.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), rep_dim=64, dropout=0.0):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.encoder = nn.Sequential(*layers, nn.Linear(d, rep_dim))
        self.head = nn.Linear(rep_dim, n_classes)

    def forward(self, x):
        z = self.encoder(x)
        return self.head(z), z


class CNN(nn.Module):
    def __init__(self, in_ch, n_classes, rep_dim=64, dropout=0.0, wide=False):
        super().__init__()
        def block(ci, co):
            return [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU()]
        if not wide:   # MNIST / Fashion-MNIST / synthetic-as-image
            feats = block(in_ch, 32) + block(32, 64) + [nn.MaxPool2d(2)] + block(64, 128) + [nn.MaxPool2d(2)]
            last = 128
        else:          # CIFAR-10
            feats = block(in_ch, 64) + block(64, 128) + [nn.MaxPool2d(2)] + block(128, 256) + [nn.MaxPool2d(2)] + block(256, 256)
            last = 256
        self.features = nn.Sequential(*feats, nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.rep = nn.Sequential(nn.Linear(last, rep_dim), nn.ReLU())
        self.head = nn.Linear(rep_dim, n_classes)

    def forward(self, x):
        z = self.rep(self.drop(self.features(x)))
        return self.head(z), z


def build_model(arch: str, data: DataBundle, dropout: float, rep_dim: Optional[int] = None) -> nn.Module:
    cifar = data.name == "cifar10"
    rd = rep_dim or (128 if cifar else 64)
    if arch == "mlp":
        hidden = (512, 256, 128) if cifar else (256, 128)
        return MLP(int(np.prod(data.in_shape)), data.n_classes, hidden, rd, dropout)
    return CNN(data.in_shape[0], data.n_classes, rd, dropout, wide=cifar)

## 4. Methods and the training loop

The seven configurations from Section 5.1. Jacobian regularisation uses the random-projection estimator of the Frobenius norm of the representation Jacobian (one random direction per step), which is the standard cheap estimator. IDRR-Adaptive anneals lambda from 0.2 to 0.02 on a cosine schedule.

Training: AdamW at 1e-3, cosine annealing over `MAX_EPOCHS`, batch 128, early stopping with patience 15 on validation accuracy, restoring the best weights.

In [ ]:
@dataclass
class Method:
    name: str
    weight_decay: float = 0.0
    dropout: float = 0.0
    jacobian: float = 0.0
    idrr_lambda: float = 0.0
    idrr_adaptive: bool = False
    idrr_target: Optional[float] = None   # override the data-driven target (ablations)


METHODS = [
    Method("Standard"),
    Method("Weight Decay", weight_decay=1e-3),
    Method("Dropout", dropout=0.3),
    Method("Jacobian Reg.", jacobian=0.01),
    Method("IDRR", idrr_lambda=0.1),
    Method("IDRR-Adaptive", idrr_lambda=0.2, idrr_adaptive=True),
    Method("IDRR+Dropout", idrr_lambda=0.05, dropout=0.3),
]
OURS = {"IDRR", "IDRR-Adaptive", "IDRR+Dropout"}


def jacobian_penalty(x, z):
    """Random-projection estimate of ||dz/dx||_F^2 / batch (Hoffman et al. 2019)."""
    v = torch.randn_like(z)
    v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
    (jv,) = torch.autograd.grad((z * v).sum(), x, create_graph=True)
    return z.shape[1] * (jv ** 2).sum() / x.shape[0]


@torch.no_grad()
def evaluate(model, x, y, batch=1024):
    model.eval()
    correct, zs = 0, []
    for i in range(0, len(x), batch):
        xb, yb = x[i:i + batch].to(DEVICE), y[i:i + batch].to(DEVICE)
        logits, z = model(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        zs.append(z.float().cpu())
    Z = torch.cat(zs)
    return 100.0 * correct / len(x), soft_effective_rank(Z).item(), Z


def train_one(data: DataBundle, method: Method, arch: str, seed: int, rep_dim=None,
              max_epochs=None, patience=None, log_dynamics=False, verbose=False):
    max_epochs = max_epochs or MAX_EPOCHS
    patience = patience or PATIENCE
    seed_everything(seed)
    model = build_model(arch, data, method.dropout, rep_dim).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=method.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    loader = DataLoader(TensorDataset(data.xtr, data.ytr), batch_size=128, shuffle=True, drop_last=True,
                        generator=torch.Generator().manual_seed(seed))
    d_target = method.idrr_target if method.idrr_target is not None else data.d_target
    best_val, best_state, bad, history = -1.0, None, 0, []
    t0 = time.time()
    for epoch in range(max_epochs):
        model.train()
        lam = method.idrr_lambda
        if method.idrr_adaptive:
            lam = 0.02 + 0.5 * (0.2 - 0.02) * (1 + math.cos(math.pi * epoch / max_epochs))
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if method.jacobian > 0:
                xb.requires_grad_(True)
            logits, z = model(xb)
            loss = F.cross_entropy(logits, yb)
            if lam > 0:
                loss = loss + lam * idrr_loss(soft_effective_rank(z), d_target)
            if method.jacobian > 0:
                loss = loss + method.jacobian * jacobian_penalty(xb, z)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        sched.step()
        val_acc, _, _ = evaluate(model, data.xva, data.yva)
        if log_dynamics:
            tr_acc, _, _ = evaluate(model, data.xtr[:2000], data.ytr[:2000])
            te_acc, te_rank, _ = evaluate(model, data.xte, data.yte)
            history.append({"epoch": epoch, "train_acc": tr_acc, "test_acc": te_acc, "erank": te_rank, "gap": tr_acc - te_acc})
        if val_acc > best_val:
            best_val, bad = val_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose and epoch % 10 == 0:
            print(f"  epoch {epoch:3d} val {val_acc:.2f} best {best_val:.2f}")
    model.load_state_dict(best_state)
    test_acc, test_rank, Z = evaluate(model, data.xte, data.yte)
    return {"dataset": data.name, "method": method.name, "arch": arch, "seed": seed, "acc": test_acc,
            "erank": test_rank, "d_target": d_target, "epochs": epoch + 1, "secs": time.time() - t0,
            "history": history, "Z": Z, "y": data.yte.clone()}

Quick check on synthetic data with one seed so we know the loop works before the long sweeps.

In [ ]:
_d = get_data("synthetic", 0, flatten=True)
print("synthetic d_target =", round(_d.d_target, 2))
for m in [METHODS[0], METHODS[6]]:
    r = train_one(_d, m, "mlp", 0, max_epochs=8 if RUN_MODE == "smoke" else 40, patience=5)
    print(f"{m.name:14s} acc {r['acc']:.1f}  erank {r['erank']:.1f}  ({r['epochs']} epochs, {r['secs']:.0f}s)")

## 5. Main sweep: Table 1 (MLP) and Table 2 (CNN)

In [ ]:
def ci95(vals):
    vals = np.asarray(vals, dtype=float)
    if len(vals) < 2:
        return float(vals.mean()), 0.0
    return float(vals.mean()), float(st.t.ppf(0.975, len(vals) - 1) * vals.std(ddof=1) / math.sqrt(len(vals)))


def run_sweep(arch: str, datasets_list, methods=METHODS, log_first_seed=True):
    rows, raw = [], {}
    flatten = arch == "mlp"
    for dname in datasets_list:
        for seed in SEEDS[dname]:
            data = get_data(dname, seed, flatten=flatten)
            for m in methods:
                r = train_one(data, m, arch, seed, log_dynamics=(log_first_seed and seed == SEEDS[dname][0]))
                raw[(dname, m.name, seed)] = r
                rows.append({k: r[k] for k in ["dataset", "method", "arch", "seed", "acc", "erank", "d_target", "epochs", "secs"]})
                print(f"[{arch}] {dname:13s} seed {seed} {m.name:14s} acc {r['acc']:5.1f} erank {r['erank']:6.1f} target {r['d_target']:5.1f} ({r['epochs']} ep, {r['secs']:.0f}s)")
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT}/raw_{arch}.csv", index=False)
    return df, raw


def summarise(df: pd.DataFrame, datasets_list):
    table = {}
    for m in df.method.unique():
        row = {}
        for d in datasets_list:
            sub = df[(df.method == m) & (df.dataset == d)]
            am, ac = ci95(sub.acc)
            rm, _ = ci95(sub.erank)
            row[(d, "Acc.")] = f"{am:.1f}±{ac:.1f}"
            row[(d, "Rank")] = f"{rm:.1f}"
        table[m] = row
    out = pd.DataFrame(table).T
    out.columns = pd.MultiIndex.from_tuples(out.columns)
    return out


DATASETS = ["synthetic", "mnist", "fashion_mnist", "cifar10"]
df_mlp, raw_mlp = run_sweep("mlp", DATASETS)
table1 = summarise(df_mlp, DATASETS)
table1.to_csv(f"{OUT}/table1_mlp.csv")
print("\nTable 1: MLP test accuracy (%) and effective rank, mean ± 95% CI")
print(table1.to_string())

In [ ]:
if RUN_CNN:
    df_cnn, raw_cnn = run_sweep("cnn", DATASETS)
    table2 = summarise(df_cnn, DATASETS)
    table2.to_csv(f"{OUT}/table2_cnn.csv")
    print("\nTable 2: CNN test accuracy (%) and effective rank, mean ± 95% CI")
    print(table2.to_string())

## 6. Statistical significance (Table 3)

Paired t-tests of IDRR+Dropout against Standard, Weight Decay and Dropout, pairing runs by seed.

In [ ]:
def paired_tests(df, datasets_list, ours="IDRR+Dropout", baselines=("Standard", "Weight Decay", "Dropout")):
    rows = {}
    for b in baselines:
        row = {}
        for d in datasets_list:
            a = df[(df.method == ours) & (df.dataset == d)].sort_values("seed").acc.values
            c = df[(df.method == b) & (df.dataset == d)].sort_values("seed").acc.values
            n = min(len(a), len(c))
            if n < 2:
                row[d] = "n/a"
                continue
            p = st.ttest_rel(a[:n], c[:n]).pvalue
            row[d] = f"{p:.4f}" + (" *" if p < 0.05 else "")
        rows[f"vs. {b}"] = row
    return pd.DataFrame(rows).T

table3 = paired_tests(df_mlp, DATASETS)
table3.to_csv(f"{OUT}/table3_pvalues.csv")
print("Table 3: paired t-test p-values, IDRR+Dropout vs baselines (* = p < 0.05)")
print(table3.to_string())

## 7. Geometry: PCA skeletons (Figure 1) and singular value decay (Figure 2)

In [ ]:
def pca2(Z):
    Zc = Z - Z.mean(0)
    U, S, Vt = torch.linalg.svd(Zc, full_matrices=False)
    return (Zc @ Vt[:2].T).numpy()


def figure1(raw, datasets_list=("mnist", "fashion_mnist"), methods=("Standard", "Dropout", "IDRR+Dropout")):
    fig, axes = plt.subplots(len(datasets_list), len(methods), figsize=(4 * len(methods), 4 * len(datasets_list)))
    axes = np.atleast_2d(axes)
    for i, d in enumerate(datasets_list):
        seed = SEEDS[d][0]
        for j, m in enumerate(methods):
            r = raw[(d, m, seed)]
            P = pca2(r["Z"])
            axes[i, j].scatter(P[:, 0], P[:, 1], c=r["y"].numpy(), cmap="tab10", s=3, alpha=0.6)
            axes[i, j].set_title(f"{d.upper()} - {m}", fontsize=10)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
            axes[i, j].set_xlabel("PCA 1"); axes[i, j].set_ylabel("PCA 2")
    plt.tight_layout(); plt.savefig(f"{OUT}/figure1_pca.png", dpi=150); plt.show()


def figure2(raw, datasets_list=("mnist", "fashion_mnist"), methods=("Standard", "Dropout", "IDRR+Dropout")):
    fig, axes = plt.subplots(1, len(datasets_list), figsize=(5 * len(datasets_list), 4))
    axes = np.atleast_1d(axes)
    for ax, d in zip(axes, datasets_list):
        seed = SEEDS[d][0]
        for m, col in zip(methods, ["tab:blue", "tab:orange", "tab:green"]):
            Z = raw[(d, m, seed)]["Z"]
            s = torch.linalg.svdvals(Z - Z.mean(0)).numpy()
            ax.semilogy(np.arange(1, len(s) + 1), s / s[0], label=m, color=col)
        ax.set_title(f"SV spectrum decay: {d.upper()}"); ax.set_xlabel("singular value index")
        ax.set_ylabel("normalised singular value"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/figure2_sv_decay.png", dpi=150); plt.show()

figure1(raw_mlp)
figure2(raw_mlp)

## 8. Training dynamics (Figures 3, 5, 6, 7) and rank comparison (Figure 4)

In [ ]:
def figure_dynamics(raw, d, methods=("Standard", "Dropout", "IDRR", "IDRR+Dropout")):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    cols = dict(zip(methods, ["tab:blue", "tab:green", "tab:purple", "tab:pink"]))
    for m in methods:
        h = pd.DataFrame(raw[(d, m, SEEDS[d][0])]["history"])
        if h.empty:
            continue
        axes[0].plot(h.epoch, h.test_acc, label=m, color=cols[m])
        axes[1].plot(h.epoch, h.erank, label=m, color=cols[m])
        axes[2].plot(h.epoch, h.gap, label=m, color=cols[m])
    for ax, t in zip(axes, ["Test accuracy (%)", "Representation effective rank", "Generalization gap (%)"]):
        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle(f"Training dynamics: {d.upper()}")
    plt.tight_layout(); plt.savefig(f"{OUT}/dynamics_{d}.png", dpi=150); plt.show()


def figure4(df, datasets_list):
    methods = list(df.method.unique())
    x = np.arange(len(datasets_list)); w = 0.8 / len(methods)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for i, m in enumerate(methods):
        means, errs = [], []
        for d in datasets_list:
            mu, e = ci95(df[(df.method == m) & (df.dataset == d)].erank)
            means.append(mu); errs.append(e)
        ax.bar(x + i * w - 0.4 + w / 2, means, w, yerr=errs, label=m, capsize=2)
    ax.set_xticks(x); ax.set_xticklabels([d.upper() for d in datasets_list])
    ax.set_ylabel("effective rank"); ax.set_title("Representation effective rank (lower = more compressed)")
    ax.legend(ncol=2, fontsize=8); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/figure4_rank_bars.png", dpi=150); plt.show()

for d in DATASETS:
    figure_dynamics(raw_mlp, d)
figure4(df_mlp, DATASETS)

## 9. Ablations on synthetic data (Table 4): lambda sensitivity and target rank sensitivity

In [ ]:
ABL_SEEDS = SEEDS["synthetic"][:3] if RUN_MODE == "full" else SEEDS["synthetic"]
lam_rows, tgt_rows = [], []
for seed in ABL_SEEDS:
    data = get_data("synthetic", seed, flatten=True)
    for lam in [0.01, 0.02, 0.05, 0.10, 0.20, 0.30]:
        r = train_one(data, Method(f"lam={lam}", idrr_lambda=lam), "mlp", seed)
        lam_rows.append({"lambda": lam, "seed": seed, "acc": r["acc"], "erank": r["erank"]})
    for tgt in [4, 8, 12, 15, 20, 25]:
        r = train_one(data, Method(f"target={tgt}", idrr_lambda=0.1, idrr_target=float(tgt)), "mlp", seed)
        tgt_rows.append({"target": tgt, "seed": seed, "acc": r["acc"], "erank": r["erank"]})

lam_df = pd.DataFrame(lam_rows).groupby("lambda")[["acc", "erank"]].mean().round(1)
tgt_df = pd.DataFrame(tgt_rows).groupby("target")[["acc", "erank"]].mean().round(1)
lam_df.to_csv(f"{OUT}/table4_lambda.csv"); tgt_df.to_csv(f"{OUT}/table4_target.csv")
print("Table 4 (left): lambda sensitivity\n", lam_df.to_string())
print("\nTable 4 (right): target rank sensitivity (achieved = erank)\n", tgt_df.to_string())

## 10. Bottleneck baselines on MNIST (Table 5)

Fixed low-dimensional representation layers trained with the Standard objective, against IDRR with rep_dim 64.

In [ ]:
bn_rows = []
for seed in SEEDS["mnist"][:3] if RUN_MODE == "full" else SEEDS["mnist"]:
    data = get_data("mnist", seed, flatten=True)
    for rd in [64, 32, 16, 12, 8]:
        r = train_one(data, Method(f"Bottleneck-{rd}"), "mlp", seed, rep_dim=rd)
        bn_rows.append({"method": f"Bottleneck-{rd}", "seed": seed, "acc": r["acc"], "erank": r["erank"]})
    r = train_one(data, Method("IDRR (rep_dim=64)", idrr_lambda=0.1), "mlp", seed, rep_dim=64)
    bn_rows.append({"method": "IDRR (rep_dim=64)", "seed": seed, "acc": r["acc"], "erank": r["erank"]})
table5 = pd.DataFrame(bn_rows).groupby("method", sort=False)[["acc", "erank"]].mean().round(1)
table5.to_csv(f"{OUT}/table5_bottleneck.csv")
print("Table 5: bottleneck baselines on MNIST\n", table5.to_string())

## 11. Key findings, computed from this run

The paper's three headline claims, checked against the numbers above rather than restated.

In [ ]:
def headline(df, datasets_list):
    for d in datasets_list:
        drop = df[(df.method == "Dropout") & (df.dataset == d)]
        ours = df[(df.method == "IDRR+Dropout") & (df.dataset == d)]
        if len(drop) == 0 or len(ours) == 0:
            continue
        red = 100 * (1 - ours.erank.mean() / drop.erank.mean())
        print(f"{d:13s} Dropout acc {drop.acc.mean():5.1f} rank {drop.erank.mean():5.1f} | "
              f"IDRR+Dropout acc {ours.acc.mean():5.1f} rank {ours.erank.mean():5.1f} | "
              f"rank reduction {red:4.0f}%  compression {drop.erank.mean() / ours.erank.mean():.1f}x")
headline(df_mlp, DATASETS)
print("\nAll CSVs and figures are in", os.path.abspath(OUT))